## Converting PDFs to Markdown

In [1]:
import os
import re
import pandas as pd
import requests

CSV_URL = "https://raw.githubusercontent.com/alexeygrigorev/ai-engineering-buildcamp-code/main/01-foundation/homework/books.csv"
DOWNLOAD_DIR = "pdf_books"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

df = pd.read_csv(CSV_URL)

def safe_filename(title):
    title = re.sub(r"[^a-zA-Z0-9_-]+", "_", title).strip("_")
    return f"{title}.pdf"

for _, row in df.iterrows():
    title = row["title"]
    url = row["pdf_url"]
    filename = safe_filename(title)
    filepath = os.path.join(DOWNLOAD_DIR, filename)

    if os.path.exists(filepath):
        print(f"Already exists: {filename}")
        continue

    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        with open(filepath, "wb") as f:
            f.write(response.content)

        print(f"Downloaded: {filename}")

    except Exception as e:
        print(f"Failed to download {title}: {e}")

Downloaded: Think_Python_2e.pdf
Downloaded: Think_DSP.pdf
Downloaded: Think_Complexity_2e.pdf
Downloaded: Think_Java_2e.pdf
Downloaded: Physical_Modeling_in_MATLAB.pdf
Downloaded: Think_OS.pdf
Downloaded: Think_C.pdf


In [3]:
!uv add 'markitdown[pdf]'

Resolved 238 packages in 978ms                                       
Prepared 9 packages in 1.22s                                             
Installed 9 packages in 13ms                                
 + flatbuffers==25.12.19
 + magika==0.6.2
 + markdownify==1.2.2
 + markitdown==0.1.5
 + onnxruntime==1.25.1
 + pdfminer-six==20251230
 + pdfplumber==0.11.9
 + pillow==12.2.0
 + pypdfium2==5.8.0


In [9]:
!markitdown books_pdf/Think_Python_2e.pdf > books_text/Think_Python_2e.md
!markitdown books_pdf/Think_DSP.pdf > books_text/Think_DSP.md
!markitdown books_pdf/Think_Complexity_2e.pdf > books_text/Think_Complexity_2e.md
!markitdown books_pdf/Think_Java_2e.pdf > books_text/Think_Java_2e.md
!markitdown books_pdf/Physical_Modeling_in_MATLAB.pdf > books_text/Physical_Modeling_in_MATLAB.md
!markitdown books_pdf/Think_OS.pdf > books_text/Think_OS.md
!markitdown books_pdf/Think_C.pdf > books_text/Think_C.md

**How many lines are in the extracted content from the "Think Python" book?**

In [10]:
!wc -l books_text/Think_Python_2e.md

   16269 books_text/Think_Python_2e.md


## Chunking for RAG

In [20]:
from pathlib import Path

books_dir = Path("books_text")

books = []

for md_file in books_dir.glob("*.md"):
    with open(md_file, "r", encoding="utf-8") as file:
        lines = file.read().splitlines()

    non_empty_lines = [
        line for line in lines
        if line.strip()
    ]

    book_dict = {
        "source": md_file.name,
        "content": non_empty_lines
    }

    books.append(book_dict)

books

[{'source': 'Think_Complexity_2e.md',
  'content': ['Think Complexity',
   'Version 2.6.3',
   'Think Complexity',
   'Version 2.6.3',
   'Allen B. Downey',
   'Green Tea Press',
   'Needham, Massachusetts',
   '| Copyright | 2016 | Allen B. | Downey. |     |',
   '| --------- | ---- | -------- | ------- | --- |',
   "'",
   '| Green Tea  | Press    |     |     |     |',
   '| ---------- | -------- | --- | --- | --- |',
   '| 9 Washburn | Ave      |     |     |     |',
   '| Needham    | MA 02492 |     |     |     |',
   'Permissionisgrantedtocopy, distribute, transmitandadaptthisworkundera',
   'Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International',
   '| License: | http://thinkcomplex.com/license. |     |     |     |',
   '| -------- | -------------------------------- | --- | --- | --- |',
   'If you are interested in distributing a commercial version of this work, please',
   '| contact   | the author. |               |              |      |',
   '| --------- | --

In [21]:
!uv add gitsource

Resolved 238 packages in 5ms
Checked 231 packages in 34ms


In [26]:
from gitsource import chunk_documents

chunks = chunk_documents(books, size=100, step=50)

chunks

[{'start': 0,
  'content': ['Think Complexity',
   'Version 2.6.3',
   'Think Complexity',
   'Version 2.6.3',
   'Allen B. Downey',
   'Green Tea Press',
   'Needham, Massachusetts',
   '| Copyright | 2016 | Allen B. | Downey. |     |',
   '| --------- | ---- | -------- | ------- | --- |',
   "'",
   '| Green Tea  | Press    |     |     |     |',
   '| ---------- | -------- | --- | --- | --- |',
   '| 9 Washburn | Ave      |     |     |     |',
   '| Needham    | MA 02492 |     |     |     |',
   'Permissionisgrantedtocopy, distribute, transmitandadaptthisworkundera',
   'Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International',
   '| License: | http://thinkcomplex.com/license. |     |     |     |',
   '| -------- | -------------------------------- | --- | --- | --- |',
   'If you are interested in distributing a commercial version of this work, please',
   '| contact   | the author. |               |              |      |',
   '| --------- | ----------- | ------------